In [1]:
import json
from collections import Counter

# ============================================================
# DATASET PATH
# ============================================================

DATASET_PATH = "./dataset/V1A.jsonl"

# Change this path
# DATASET_PATH = "./newV4a/finalV4A.jsonl"
# Change this path
# DATASET_PATH = r"G:\V5-dataset\V5A\V5A.jsonl"
#

# 
# DATASET_PATH = r"G:\V5-dataset\V5A\V5A.jsonl"
# ============================================================
# COUNTER
# ============================================================

counter = Counter()
total = 0


# ============================================================
# MESSAGE COUNTER
# ============================================================

def count_messages(conversation):
    """
    Count the number of chat messages.
    """

    count = 0

    for line in conversation.splitlines():

        line = line.strip()

        if not line:
            continue

        if ":" in line:
            count += 1

    return count


# ============================================================
# READ DATASET
# ============================================================

with open(DATASET_PATH, "r", encoding="utf-8") as f:

    for line in f:

        if not line.strip():
            continue

        sample = json.loads(line)

        messages = count_messages(
            sample["input"]["conversation"]
        )

        total += 1


        # ====================================================
        # CONVERSATION LENGTH CATEGORIES
        # ====================================================

        if 1 <= messages <= 2:
            counter["Extra Small (1-2)"] += 1

        elif 3 <= messages <= 6:
            counter["Small (3-6)"] += 1

        elif 7 <= messages <= 15:
            counter["Medium (7-15)"] += 1

        elif 16 <= messages <= 30:
            counter["Large (16-30)"] += 1

        elif messages >= 31:
            counter["Extra Large (31+)"] += 1

        else:
            counter["Other"] += 1


# ============================================================
# OUTPUT
# ============================================================

print("=" * 60)
print("CONVERSATION LENGTH ANALYSIS")
print("=" * 60)

print(f"Total Samples : {total}")

print("=" * 60)

categories = [
    "Extra Small (1-2)",
    "Small (3-6)",
    "Medium (7-15)",
    "Large (16-30)",
    "Extra Large (31+)",
    "Other"
]

for category in categories:

    count = counter[category]

    percentage = (
        count / total * 100
        if total
        else 0
    )

    print(
        f"{category:20}: "
        f"{count:6} "
        f"({percentage:6.2f}%)"
    )

print("=" * 60)

CONVERSATION LENGTH ANALYSIS
Total Samples : 48114
Extra Small (1-2)   :     21 (  0.04%)
Small (3-6)         :  11037 ( 22.94%)
Medium (7-15)       :  30006 ( 62.36%)
Large (16-30)       :   7050 ( 14.65%)
Extra Large (31+)   :      0 (  0.00%)
Other               :      0 (  0.00%)


In [ ]:
# ======================================================================
# V5C CONVERSATION LENGTH ANALYSIS
# ======================================================================
# Dataset          : G:\V5-dataset\V5C\V5C.jsonl
# Total records    : 62,493
# Valid records    : 62,493
# Invalid JSON     : 0
# Missing input    : 0
# ======================================================================
# CONVERSATION LENGTH DISTRIBUTION
# ======================================================================
# Extra Small (1-2)     :   4,751 (  7.60%)
# Small (3-6)           :   5,905 (  9.45%)
# Medium (7-15)         :  48,228 ( 77.17%)
# Large (16-30)         :   3,609 (  5.78%)
# Extra Large (31+)     :       0 (  0.00%)
# Other                 :       0 (  0.00%)
# ======================================================================
# PROBLEM RECORDS
# ======================================================================
# Total problems : 0
# ✓ No problematic records found.
# ======================================================================
# FINAL STATUS
# ======================================================================
# ✓ ALL RECORDS HAVE A VALID CONVERSATION FIELD.
# ======================================================================

Code For removeing extra small sample

In [ ]:
import json

# ============================================================
# PATHS
# ============================================================

INPUT_PATH = r"G:\V5-dataset\V5C\V5C.jsonl"

OUTPUT_PATH = r"G:\V5-dataset\V5C\V5C_no_extra_small.jsonl"


# ============================================================
# COUNT MESSAGES
# ============================================================

def count_messages(conversation):
    count = 0

    for line in conversation.splitlines():
        line = line.strip()

        if not line:
            continue

        if ":" in line:
            count += 1

    return count


# ============================================================
# REMOVE EXTRA SMALL
# Extra Small = 1–2 messages
# Keep = 3+ messages
# ============================================================

total = 0
removed = 0
kept = 0

with open(INPUT_PATH, "r", encoding="utf-8") as infile, \
     open(OUTPUT_PATH, "w", encoding="utf-8") as outfile:

    for line in infile:

        if not line.strip():
            continue

        total += 1

        sample = json.loads(line)

        conversation = sample["input"]["conversation"]

        message_count = count_messages(conversation)

        # Remove 1–2 message conversations
        if message_count <= 2:
            removed += 1
            continue

        # Keep 3+ message conversations
        outfile.write(
            json.dumps(sample, ensure_ascii=False) + "\n"
        )

        kept += 1


# ============================================================
# RESULT
# ============================================================

print("=" * 70)
print("V5C EXTRA SMALL SAMPLE REMOVAL")
print("=" * 70)

print(f"Original samples      : {total}")
print(f"Removed (1–2 messages): {removed}")
print(f"Remaining samples     : {kept}")

if total > 0:
    print(f"Removed percentage    : {(removed / total) * 100:.2f}%")
    print(f"Remaining percentage  : {(kept / total) * 100:.2f}%")

print("=" * 70)
print("OUTPUT FILE:")
print(OUTPUT_PATH)
print("=" * 70)

V5C EXTRA SMALL SAMPLE REMOVAL
Original samples      : 64216
Removed (1–2 messages): 29687
Remaining samples     : 34529
Removed percentage    : 46.23%
Remaining percentage  : 53.77%
OUTPUT FILE:
G:\V5-dataset\V5C\V5C_no_extra_small.jsonl


In [ ]:
import json

# ============================================================
# PATHS
# ============================================================

INPUT_PATH = r"G:\V5-dataset\V5C-Bittu.jsonl"

# Main output: keeps conversations with 3+ messages
OUTPUT_PATH = r"G:\V5-dataset\V5C\V5C_no_extra_small.jsonl"

# Separate output: stores removed 1–2 message conversations
REMOVED_PATH = r"G:\V5-dataset\V5C\V5C_extra_small.jsonl"


# ============================================================
# COUNT MESSAGES
# ============================================================

def count_messages(conversation):
    count = 0

    for line in conversation.splitlines():
        line = line.strip()

        if not line:
            continue

        # Count lines containing ":" as messages
        if ":" in line:
            count += 1

    return count


# ============================================================
# PROCESS DATASET
#
# Extra Small = 1–2 messages
# Keep        = 3+ messages
#
# V5C_no_extra_small.jsonl
#     -> contains ONLY 3+ message conversations
#
# V5C_extra_small.jsonl
#     -> contains ONLY 1–2 message conversations
# ============================================================

total = 0
removed = 0
kept = 0

with open(INPUT_PATH, "r", encoding="utf-8") as infile, \
     open(OUTPUT_PATH, "w", encoding="utf-8") as outfile, \
     open(REMOVED_PATH, "w", encoding="utf-8") as removedfile:

    for line_number, line in enumerate(infile, start=1):

        # Skip completely blank lines
        if not line.strip():
            continue

        try:
            sample = json.loads(line)

        except json.JSONDecodeError as e:
            print(f"WARNING: Invalid JSON on line {line_number}")
            print(f"Error: {e}")
            continue

        total += 1

        # ----------------------------------------------------
        # Get conversation
        # ----------------------------------------------------

        try:
            conversation = sample["input"]["conversation"]

        except KeyError:
            print(
                f"WARNING: Missing input/conversation "
                f"on line {line_number}"
            )
            continue

        # ----------------------------------------------------
        # Count messages
        # ----------------------------------------------------

        message_count = count_messages(conversation)

        # ----------------------------------------------------
        # EXTRA SMALL: 1–2 messages
        # Save to separate file
        # ----------------------------------------------------

        if message_count <= 2:

            removedfile.write(
                json.dumps(
                    sample,
                    ensure_ascii=False
                ) + "\n"
            )

            removed += 1

        # ----------------------------------------------------
        # KEEP: 3+ messages
        # Save to main output
        # ----------------------------------------------------

        else:

            outfile.write(
                json.dumps(
                    sample,
                    ensure_ascii=False
                ) + "\n"
            )

            kept += 1


# ============================================================
# RESULT
# ============================================================

print("=" * 70)
print("V5C EXTRA SMALL SAMPLE REMOVAL")
print("=" * 70)

print(f"Original samples       : {total}")
print(f"Removed (1–2 messages) : {removed}")
print(f"Remaining (3+ messages): {kept}")

print("-" * 70)

if total > 0:
    print(
        f"Removed percentage     : "
        f"{(removed / total) * 100:.2f}%"
    )

    print(
        f"Remaining percentage   : "
        f"{(kept / total) * 100:.2f}%"
    )

print("=" * 70)

print("MAIN OUTPUT FILE:")
print(OUTPUT_PATH)

print("-" * 70)

print("REMOVED EXTRA SMALL FILE:")
print(REMOVED_PATH)

print("=" * 70)

V5C EXTRA SMALL SAMPLE REMOVAL
Original samples       : 64374
Removed (1–2 messages) : 29742
Remaining (3+ messages): 34632
----------------------------------------------------------------------
Removed percentage     : 46.20%
Remaining percentage   : 53.80%
MAIN OUTPUT FILE:
G:\V5-dataset\V5C\V5C_no_extra_small.jsonl
----------------------------------------------------------------------
REMOVED EXTRA SMALL FILE:
G:\V5-dataset\V5C\V5C_extra_small.jsonl
